In [1]:
import os
import json 
import argparse 
from typing import Dict, List, Any

In [2]:
try: 
    from datasets import load_dataset 
except ImportError: 
    raise ImportError("Please install Hugging Face datasets: pip install datasets")

In [3]:
def extract_xquad(num_samples: int = 100) -> List[Dict[str, Any]]: 
    """ 
    
    Extracts parallel XQuAD items across Turkish (xquad.tr), English (xquad.en), and German (xquad.de). 
    
    """ 
    
    print("Loading XQuAD splits (xquad.tr, xquad.en, xquad.de)...") 
    ds_tr = load_dataset("xquad", "xquad.tr", split="validation") 
    ds_en = load_dataset("xquad", "xquad.en", split="validation") 
    ds_de = load_dataset("xquad", "xquad.de", split="validation") 
    
    extracted = [] 
    limit = min(num_samples, len(ds_tr), len(ds_en), len(ds_de)) 
    
    for i in range(limit): 
        item_tr = ds_tr[i] 
        item_en = ds_en[i] 
        item_de = ds_de[i] 
        
        extracted.append({ 
            "id": f"xquad_{i}", 
            "task": "SpanQA", 
            "turkish": { 
                "context": item_tr["context"], 
                "question": item_tr["question"], 
                "answers": item_tr["answers"]["text"] 
            }, 
            "english": { 
                "context": item_en["context"], 
                "question": item_en["question"], 
                "answers": item_en["answers"]["text"] 
            }, 
            "german": { 
                "context": item_de["context"], 
                "question": item_de["question"], 
                "answers": item_de["answers"]["text"] 
            } 
        }) 
        
    print(f"Extracted {len(extracted)} parallel XQuAD items.") 
        
    return extracted

In [4]:
def extract_xnli(num_samples: int = 100) -> List[Dict[str, Any]]: 
    """ 
    
    Extracts parallel XNLI items across Turkish (tr), English (en), and German (de). 
    Labels: 0 = entailment, 1 = neutral, 2 = contradiction. 
    
    """ 
    
    print("Loading XNLI splits (tr, en, de)...") 
    
    ds_tr = load_dataset("xnli", "tr", split="validation") 
    ds_en = load_dataset("xnli", "en", split="validation") 
    ds_de = load_dataset("xnli", "de", split="validation") 
    
    extracted = [] 
    limit = min(num_samples, len(ds_tr), len(ds_en), len(ds_de)) 
    
    for i in range(limit): 
        item_tr = ds_tr[i] 
        item_en = ds_en[i] 
        item_de = ds_de[i] 
        
        extracted.append({ 
            "id": f"xnli_{i}", 
            "task": "NLI", 
            "label": item_tr["label"], # 0,1, or 2
            "turkish": { 
                "premise": item_tr["premise"], 
                "hypothesis": item_tr["hypothesis"] 
            }, 
            "english": { 
                "premise": item_en["premise"], 
                "hypothesis": item_en["hypothesis"] 
            }, 
            "german": { 
                "premise": item_de["premise"], 
                "hypothesis": item_de["hypothesis"] 
            } 
        }) 
        
    print(f"Extracted {len(extracted)} parallel XNLI items.") 
        
    return extracted

In [5]:
def extract_belebele(num_samples: int = 100) -> List[Dict[str, Any]]:
    """
    
    Extract parallel Belebele reading comprehension items (tur_Latn, eng_Latn, deu_Latn).
    
    """
   
    print("Loading Belebele splits (tur_Latn, eng_Latn, deu_Latn)...")

    ds_tr = load_dataset("facebook/belebele", "tur_Latn", split="test")
    ds_en = load_dataset("facebook/belebele", "eng_Latn", split="test")
    ds_de = load_dataset("facebook/belebele", "deu_Latn", split="test")

    extracted: List[Dict[str, Any]] = []
    limit = min(num_samples, len(ds_tr), len(ds_en), len(ds_de))

    for i in range(limit):
        item_tr, item_en, item_de = ds_tr[i], ds_en[i], ds_de[i]
        extracted.append(
            {
                "id": f"belebele_{i}",
                "task": "ReadingComprehensionReasoning",
                "link": item_tr.get("link", f"{i}"),
                "correct_answer_num": item_tr["correct_answer_num"], # "1", "2", "3", or "4"
                "turkish": {
                    "passage": item_tr["flores_passage"],
                    "question": item_tr["question"],
                    "mc_answer1": item_tr["mc_answer1"],
                    "mc_answer2": item_tr["mc_answer2"],
                    "mc_answer3": item_tr["mc_answer3"],
                    "mc_answer4": item_tr["mc_answer4"],
                },
                "english": {
                    "passage": item_en["flores_passage"],
                    "question": item_en["question"],
                    "mc_answer1": item_en["mc_answer1"],
                    "mc_answer2": item_en["mc_answer2"],
                    "mc_answer3": item_en["mc_answer3"],
                    "mc_answer4": item_en["mc_answer4"],
                },
                "german": {
                    "passage": item_de["flores_passage"],
                    "question": item_de["question"],
                    "mc_answer1": item_de["mc_answer1"],
                    "mc_answer2": item_de["mc_answer2"],
                    "mc_answer3": item_de["mc_answer3"],
                    "mc_answer4": item_de["mc_answer4"],
                },
            }
        )

    return extracted


In [6]:
def save_json(data: List[Dict[str, Any]], filepath: str): 
    os.makedirs(os.path.dirname(filepath), exist_ok=True) 
   
    with open(filepath, "w", encoding="utf-8") as f: 
        json.dump(data, f, ensure_ascii=False, indent=2) 
    
    print(f"Saved {len(data)} items to '{filepath}'.")

In [7]:
def main(): 
    parser = argparse.ArgumentParser(description= "Extract parallel TR-EN-DE benchmark items into task-specific JSON files.")
    
    parser.add_argument("--num_samples", type=int, default=100, help="Number of items per task") 
    parser.add_argument("--output_dir", type=str, default="test_suite", help="Output directory") 
    args = parser.parse_args(args=['--num_samples', '100', '--output_dir', 'test_suite'])
    
    save_json(extract_xnli(args.num_samples), os.path.join(args.output_dir, "xnli_suite.json")) 
    save_json(extract_belebele(args.num_samples), os.path.join(args.output_dir, "belebele_suite.json")) 
    save_json(extract_xquad(args.num_samples), os.path.join(args.output_dir, "xquad_suite.json")) 
    
    print(f"\n All task suites generated inside directory '{args.output_dir}/'!")
    
if __name__ == "__main__": 
        main()

Loading XNLI splits (tr, en, de)...


README.md: 0.00B [00:00, ?B/s]

tr/train-00000-of-00001.parquet:   0%|          | 0.00/48.0M [00:00<?, ?B/s]

tr/test-00000-of-00001.parquet:   0%|          | 0.00/338k [00:00<?, ?B/s]

tr/validation-00000-of-00001.parquet:   0%|          | 0.00/172k [00:00<?, ?B/s]

...

xquad.en/validation-00000-of-00001.parqu(…):   0%|          | 0.00/212k [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/1190 [00:00<?, ? examples/s]

xquad.de/validation-00000-of-00001.parqu(…):   0%|          | 0.00/242k [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/1190 [00:00<?, ? examples/s]

Extracted 100 parallel XQuAD items.
Saved 100 items to 'test_suite/xquad_suite.json'.

 All task suites generated inside directory 'test_suite/'!
